In [1]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama
import re
from typing import Optional
import gradio as gr

import fitz  # PyMuPDF
from io import BytesIO


d:\app\Anaconda\envs\timi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "llama3.2"

In [3]:
def is_pdf_url(url: str) -> bool:
    return url.lower().endswith(".pdf")

class ContentSource:
    """
    Represents an AI paper or article to transform into content
    """

    def __init__(self, url, content_type='article'):
        self.url = url
        self.content_type = content_type

        if self.content_type == "paper" or url.endswith(".pdf"):
            self._extract_pdf()
        else:
            self._extract_html()

    def _extract_html(self):
        response = requests.get(self.url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        self.title = soup.title.string if soup.title else "No title found"

        if soup.body:
            for tag in soup.body.find_all(
                ["script", "style", "img", "input", "nav", "footer", "header"]
            ):
                tag.decompose()

            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = soup.get_text(separator="\n", strip=True)

        self.text = re.sub(r'\n\s*\n', '\n\n', self.text)

    def _extract_pdf(self):
        response = requests.get(self.url, timeout=20)
        response.raise_for_status()

        pdf = fitz.open(stream=BytesIO(response.content), filetype="pdf")

        pages_text = []
        for page in pdf:
            pages_text.append(page.get_text())

        self.text = "\n\n".join(pages_text)

        # Fallback title extraction
        self.title = self._infer_title_from_text()

    def _infer_title_from_text(self):
        lines = [l.strip() for l in self.text.split("\n") if len(l.strip()) > 10]
        return lines[0][:200] if lines else "AI Paper"


In [4]:
# Test with an AI article
print("Testing content extraction...")
test_source = ContentSource("https://www.anthropic.com/engineering/building-effective-agents")
print(test_source.title)
print(test_source.text)

Testing content extraction...
Building Effective AI Agents \ Anthropic
Engineering at Anthropic
Building effective agents
Published
Dec 19, 2024
We've worked with dozens of teams building LLM agents across industries. Consistently, the most successful implementations use simple, composable patterns rather than complex frameworks.
Over the past year, we've worked with dozens of teams building large language model (LLM) agents across industries. Consistently, the most successful implementations weren't using complex frameworks or specialized libraries. Instead, they were building with simple, composable patterns.
In this post, we share what we’ve learned from working with our customers and building agents ourselves, and give practical advice for developers on building effective agents.
What are agents?
"Agent" can be defined in several ways. Some customers define agents as fully autonomous systems that operate independently over extended periods, using various tools to accomplish complex

In [5]:
# System prompt for LinkedIn transformation
linkedin_system_prompt = """You are a LinkedIn thought leadership expert who transforms technical AI papers and articles into engaging, professional LinkedIn posts.

Your posts should:
- Start with a compelling hook that grabs attention
- Distill complex technical concepts into accessible insights
- Include 3-5 key takeaways or insights
- Use strategic line breaks for readability
- End with a thought-provoking question or call-to-action
- Be 150-300 words (LinkedIn optimal length)
- Use professional yet conversational tone
- Include relevant emojis sparingly (1-3 max)
- Focus on practical implications and industry impact

Format the post in a way that's ready to copy-paste into LinkedIn."""

In [6]:
def user_prompt_for_linkedin(content_source):
    user_prompt = f"""Transform the following {content_source.content_type} into a compelling LinkedIn thought leadership post.

Title: {content_source.title}
URL: {content_source.url}

Content:
{content_source.text[:4000]}  # Limit to avoid token limits

Create a LinkedIn post that:
1. Captures the most important insights
2. Explains why this matters to professionals
3. Engages the reader with a strong opening
4. Ends with a question or discussion prompt

Generate ONLY the LinkedIn post text, ready to publish."""
    return user_prompt

In [7]:
def messages_for_linkedin(content_source):
    """Create message format for Ollama"""
    return [
        {"role": "system", "content": linkedin_system_prompt},
        {"role": "user", "content": user_prompt_for_linkedin(content_source)}
    ]

## Use Ollama + model

In [8]:
def generate_linkedin_post(url, content_type='article'):
    """
    Generate a LinkedIn post from a URL
    Args:
        url: The URL of the paper or article
        content_type: 'paper' or 'article'
    Returns:
        Generated LinkedIn post text
    """
    print(f"Extracting content from {url}...")
    content = ContentSource(url, content_type)
    
    print(f"Generating LinkedIn post with {MODEL}...")
    messages = messages_for_linkedin(content)
    response = ollama.chat(model=MODEL, messages=messages)
    
    return response['message']['content']

In [9]:
url = "https://www.anthropic.com/engineering/building-effective-agents"
generate_linkedin_post(url)

Extracting content from https://www.anthropic.com/engineering/building-effective-agents...
Generating LinkedIn post with llama3.2...


"**Unlock the Power of Simple, Composable AI Agents**\n\nWhen it comes to building effective AI agents, simplicity is key. 🤖\n\nAt Anthropic, we've worked with dozens of teams building large language model (LLM) agents across industries. The most successful implementations use simple, composable patterns rather than complex frameworks. But why? What sets these simple patterns apart?\n\nAt its core, an agent is a system where an LLM dynamically directs its own processes and tool usage, maintaining control over how it accomplishes tasks. This distinction is crucial, as it highlights two fundamental types of agentic systems: **workflows** and **agents**.\n\n**Workflows** are systems where LLMs and tools are orchestrated through predefined code paths. **Agents**, on the other hand, are systems where LLMs dynamically direct their own processes and tool usage, maintaining control over how they accomplish tasks.\n\nSo, when should you use agents versus workflows? **Agents** trade latency and 

In [10]:
def display_linkedin_post(url, content_type='article'):
    """
    Generate and display a LinkedIn post nicely formatted
    """
    post = generate_linkedin_post(url, content_type)
    
    print("\n" + "="*60)
    print("LINKEDIN POST")
    print("="*60 + "\n")
    display(Markdown(post))
    print("\n" + "="*60)
    print(f"Ready to copy and paste into LinkedIn!")
    print("="*60)
    
    return post

In [11]:
display_linkedin_post(url)

Extracting content from https://www.anthropic.com/engineering/building-effective-agents...
Generating LinkedIn post with llama3.2...

LINKEDIN POST



**Unlock the Power of Simple, Composable AI Agents**

In today's fast-paced AI landscape, the distinction between complex frameworks and simple, composable patterns can make all the difference. At Anthropic, we've worked with dozens of teams building large language model (LLM) agents across industries, and the most successful implementations use simple, composable patterns rather than complex frameworks.

So, what sets effective agents apart? 🤔

**Agents vs. Workflows: Understanding the Architectural Distinction**

At Anthropic, we categorize agentic systems into two main types: workflows and agents. Workflows are systems where LLMs and tools are orchestrated through predefined code paths, while agents are systems where LLMs dynamically direct their own processes and tool usage.

**When to Use Agents: Unlocking Flexibility and Model-Driven Decision-Making**

Effective agents offer flexibility and model-driven decision-making at scale, making them ideal for applications where predictability and consistency are not sufficient. By using agents, you can optimize single LLM calls with retrieval and in-context examples, leading to better task performance.

**3 Key Takeaways for Building Effective Agents**

1. **Start simple**: Find the simplest solution possible, and only increase complexity when needed.
2. **Understand the underlying code**: If using a framework, ensure you understand the underlying code to avoid incorrect assumptions and common customer errors.
3. **Focus on building blocks**: The augmented LLM is the foundational building block of agentic systems, and by focusing on this, you can create more complex compositional workflows and autonomous agents.

**The Future of AI Development: What's Next?**

As AI continues to evolve, the importance of simple, composable patterns will only grow. What are your thoughts on the role of agents in AI development? Share your insights and experiences in the comments below!

#AI #MachineLearning #Agents #LLMs #Anthropic


Ready to copy and paste into LinkedIn!


"**Unlock the Power of Simple, Composable AI Agents**\n\nIn today's fast-paced AI landscape, the distinction between complex frameworks and simple, composable patterns can make all the difference. At Anthropic, we've worked with dozens of teams building large language model (LLM) agents across industries, and the most successful implementations use simple, composable patterns rather than complex frameworks.\n\nSo, what sets effective agents apart? 🤔\n\n**Agents vs. Workflows: Understanding the Architectural Distinction**\n\nAt Anthropic, we categorize agentic systems into two main types: workflows and agents. Workflows are systems where LLMs and tools are orchestrated through predefined code paths, while agents are systems where LLMs dynamically direct their own processes and tool usage.\n\n**When to Use Agents: Unlocking Flexibility and Model-Driven Decision-Making**\n\nEffective agents offer flexibility and model-driven decision-making at scale, making them ideal for applications whe

In [12]:
display_linkedin_post(
    "https://arxiv.org/pdf/1706.03762",
    content_type="paper"
)

Extracting content from https://arxiv.org/pdf/1706.03762...
Generating LinkedIn post with llama3.2...

LINKEDIN POST



**Revolutionizing Sequence Modeling: The Transformer Architecture**

Imagine being able to process entire conversations or texts without the need for repetitive calculations, freeing up vast amounts of computational resources. This is exactly what the Transformer architecture achieves, transforming the way we approach sequence modeling and transduction tasks.

In a groundbreaking paper published in 2017, the Google Research team introduced a new approach to machine translation and language modeling that dispenses with traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs). By leveraging attention mechanisms, the Transformer architecture can process sequences in parallel, making it significantly more efficient and scalable.

**So, what does this mean for professionals in the field?**

This development matters to professionals in machine learning, natural language processing, and computer vision, as it enables the creation of more efficient, scalable, and effective models. The Transformer architecture has already demonstrated impressive results in various tasks, including machine translation, language modeling, and English constituency parsing.

**Key Takeaways:**

1️⃣ The Transformer architecture is a game-changer for sequence modeling, enabling parallel processing and significant computational efficiency gains.

2️⃣ This development has far-reaching implications for industries such as healthcare, finance, and education, where natural language processing and machine translation are crucial.

3️⃣ The simplicity and elegance of the Transformer architecture make it an attractive solution for a wide range of applications, from chatbots to speech recognition systems.

4️⃣ As the field continues to evolve, it's essential to explore the practical implications of the Transformer architecture and its potential applications.

What are your thoughts on the Transformer architecture? How do you see its potential applications in your industry or organization? Share your insights in the comments below!


Ready to copy and paste into LinkedIn!


"**Revolutionizing Sequence Modeling: The Transformer Architecture**\n\nImagine being able to process entire conversations or texts without the need for repetitive calculations, freeing up vast amounts of computational resources. This is exactly what the Transformer architecture achieves, transforming the way we approach sequence modeling and transduction tasks.\n\nIn a groundbreaking paper published in 2017, the Google Research team introduced a new approach to machine translation and language modeling that dispenses with traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs). By leveraging attention mechanisms, the Transformer architecture can process sequences in parallel, making it significantly more efficient and scalable.\n\n**So, what does this mean for professionals in the field?**\n\nThis development matters to professionals in machine learning, natural language processing, and computer vision, as it enables the creation of more efficient, scalab

In [13]:
def generate_linkedin_post_gradio(url, content_type):
    try:
        if not url.strip():
            return "Please enter a valid URL."

        # Auto-detect PDF
        if url.endswith(".pdf"):
            content_type = "paper"

        content = ContentSource(url, content_type)

        messages = messages_for_linkedin(content)
        response = ollama.chat(model=MODEL, messages=messages)

        return response["message"]["content"]

    except Exception as e:
        return f"Error: {str(e)}"


In [14]:
with gr.Blocks(
    title="AI Paper → LinkedIn Post Generator",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # AI Paper → LinkedIn Thought Leadership Generator

        Paste a **paper or article URL** and get a **ready-to-publish LinkedIn post**.

        Works great with:
        - arXiv papers 
        - Blogs & research articles 
        """
    )

    with gr.Row():
        url_input = gr.Textbox(
            label="Article / Paper URL",
            placeholder="https://arxiv.org/abs/1706.03762",
            scale=4
        )

    content_type = gr.Radio(
        choices=["article", "paper"],
        value="paper",
        label="Content Type"
    )

    generate_btn = gr.Button("Generate LinkedIn Post", variant="primary")

    output = gr.Textbox(
        label="LinkedIn Post (Copy & Paste)",
        lines=15
    )

    generate_btn.click(
        fn=generate_linkedin_post_gradio,
        inputs=[url_input, content_type],
        outputs=output
    )

C:\Users\Admin\AppData\Local\Temp\ipykernel_11116\2168272591.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [15]:
demo.launch()
# demo.launch(share=True) # optional

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
